# 01 · Put the Desk Behind an API

**Module 07 · Production & Observability**

The support desk works. But today it is a Python function that only *you* can call. A real company needs it **behind an address**, so that a website, a mobile app or the email system can send it a customer email and get an answer back.

This module takes the guarded desk from Module 05 and makes it ready for real use:

| # | Notebook | The question it answers |
|---|---|---|
| 1 | **Put the desk behind an API** (this one) | How do other programs talk to it? |
| 2 | Make it reliable | What happens when things fail? |
| 3 | Logging and tracing | What happened inside one request? |
| 4 | Cost, speed and quality | How is the desk doing overall? |
| 5 | Alerts and closing the loop | How do I find out when it goes wrong, and fix it for good? |
| 6 | Going live without Docker | How do I run it for real, safely? |

> 📁 The desk is in `desk.py`, `guards.py` and `shop.py` (from Module 05, now with tracing added). The service is in `desk_service.py`.

## What is an API?
An **API** (here: a *web API*) lets one program call another over the network, using the same HTTP that browsers use.

A **request** has: a method (`GET` to read, `POST` to send something), a URL, headers (extra information, such as a key), and often a JSON body.
A **response** has a **status code** and a body.

| Status | Meaning |
|---|---|
| `200` | OK |
| `401` | Who are you? (no valid API key) |
| `403` | I know who you are, but you may not do this |
| `422` | Your request is not in the right shape |
| `429` | Too many requests |
| `500` | Something broke on the server |

We use **FastAPI**, a popular Python framework for web APIs. It reads your type hints and checks every request for you.

In [1]:
import json
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

import desk_service

# TestClient pretends to be a customer's app. It calls the service directly, so no server and no network are needed.
client = TestClient(desk_service.app)

CLIENT = {"X-API-Key": "dev-client-key"}   # what a customer's app sends
ADMIN = {"X-API-Key": "dev-admin-key"}     # what your own staff send


def show(response):
    """Print a response: its status code and its JSON body."""
    print("HTTP", response.status_code)
    print(json.dumps(response.json(), indent=2))

print("Ready")

Ready


## 1. A tiny API first
Three things to see: a `GET`, a `POST` with a body, and what FastAPI does with a body in the wrong shape.

In [2]:
tiny = FastAPI()


class Greeting(BaseModel):
    name: str


@tiny.get("/hello")
def hello():
    return {"message": "hello"}


@tiny.post("/greet")
def greet(body: Greeting):
    return {"message": "Hello, " + body.name + "!"}


tiny_client = TestClient(tiny)

show(tiny_client.get("/hello"))
show(tiny_client.post("/greet", json={"name": "Ana"}))

HTTP 200
{
  "message": "hello"
}
HTTP 200
{
  "message": "Hello, Ana!"
}


In [3]:
# A body in the wrong shape: FastAPI answers with 422 and says exactly what is wrong. We wrote no checking code.
show(tiny_client.post("/greet", json={"nom": "Ana"}))

HTTP 422
{
  "detail": [
    {
      "type": "missing",
      "loc": [
        "body",
        "name"
      ],
      "msg": "Field required",
      "input": {
        "nom": "Ana"
      }
    }
  ]
}


## 2. The support desk as an API
The service has a few parts. First, **the shape of what goes in and out**. Pydantic checks every request automatically. Notice the limits: an empty email or a gigantic body is rejected before it reaches the desk.

*From `desk_service.py`:*

```python
class EmailRequest(BaseModel):
    email: str = Field(min_length=1, max_length=20000)  # an empty or gigantic body is rejected with a 422 error
    customer_id: str = "anonymous"


class EmailResponse(BaseModel):
    request_id: str                # quote this id when you report a problem: it finds the trace
    status: str                    # replied, blocked or human
    reply: str
    route: str
    note: str
    refunds_issued: list[str]
    sent_for_approval: list[str]
    cost_usd: float
    latency_ms: int


class FeedbackRequest(BaseModel):
    request_id: str
    rating: Literal["up", "down"]
    comment: str = ""


class ApprovalRequest(BaseModel):
    approver: str = Field(min_length=1)  # who is approving? It goes in the audit trail
```

Next, **the heart of the service**: `process_email`. Read the comments. It handles one email, and *whatever happens*, it returns an answer and never crashes. If the desk fails, the email goes to a person.

It also records a **trace** of the request (the `with observe.trace(...)` block), which is what notebooks 3 to 5 are about.

*From `desk_service.py`:*

```python
def process_email(email, customer_id="anonymous"):
    """Handle one customer email. Whatever happens, this returns an answer and never raises an error."""
    with desk_lock:
        refunds_before = len(shop.REFUNDS)
        pending_before = list(guards.PENDING_APPROVALS)

        # Everything inside this block belongs to ONE trace, saved when the block ends.
        with observe.trace("support_email", customer_id=customer_id,
                           email_redacted=observe.redact_for_logs(email)[:1000]) as request:
            if state["kill_switch"]:
                # The kill switch is on: no AI at all. Every email goes straight to a person.
                result = {"status": "human", "route": "", "reply": "", "evidence": "",
                          "note": "The desk is switched off, so a person will answer this email."}
            else:
                try:
                    result = desk.handle_email(email, guarded=True)
                except Exception as error:
                    # The safe fallback: OpenAI is down, the budget ran out, a bug... The customer is not left with an error.
                    # The email goes to a person, and we log what went wrong.
                    message = type(error).__name__ + ": " + str(error)
                    observe.log(logging.ERROR, "desk_failed", error=message)
                    request.status = "error"
                    request.error = message
                    request.set(degraded=True)
                    result = {"status": "human", "route": "", "reply": "", "evidence": "",
                              "note": "The desk had a problem, so a person will answer this email."}

            # What did THIS request change? (The lists are shared by all requests, so compare before and after.)
            refunds_issued = []
            for refund in shop.REFUNDS[refunds_before:]:
                refunds_issued.append(refund["order_id"])
            sent_for_approval = []
            for order_id in guards.PENDING_APPROVALS:
                if order_id not in pending_before:
                    sent_for_approval.append(order_id)

            request.set(outcome=result["status"], route=result["route"], refunds=refunds_issued,
                        pending=sent_for_approval, note=result["note"][:200],
                        reply_redacted=observe.redact_for_logs(result["reply"])[:1000],
                        evidence_redacted=observe.redact_for_logs(result["evidence"])[:1000])

    totals = request.totals()
    return {
        "request_id": request.id,
        "status": result["status"],
        "reply": result["reply"],
        "route": result["route"],
        "note": result["note"],
        "refunds_issued": refunds_issued,
        "sent_for_approval": sent_for_approval,
        "cost_usd": totals["cost_usd"],
        "latency_ms": request.duration_ms,
        "evidence": result["evidence"],  # kept for the quality judge. It is not part of the API response.
    }
```

Last, **the endpoints**. Each one is a Python function with a decorator that says which URL and method it answers.

*From `desk_service.py`:*

```python
@app.get("/health")
def health():
    """Is the service alive? Hosting platforms call this address every few seconds."""
    return {"status": "ok", **observe.VERSIONS}


@app.post("/emails", response_model=EmailResponse)
def post_email(body: EmailRequest, background: BackgroundTasks, caller: str = Depends(require_client),
               idempotency_key: str | None = Header(default=None)):
    """Send a customer email to the desk. Add an 'Idempotency-Key' header to make retries safe."""
    if not limiter.allow(caller):
        raise HTTPException(status_code=429, detail="Too many requests. Wait a minute and try again.",
                            headers={"Retry-After": "60"})

    # Seen this exact request before? Return the saved answer and do NOTHING again (no second refund).
    saved_key = None
    if idempotency_key is not None:
        saved_key = caller + ":" + idempotency_key
        saved = idempotency.get(saved_key)
        if saved is not None:
            return saved

    outcome = process_email(body.email, body.customer_id)

    response = {}
    for field in EmailResponse.model_fields:
        response[field] = outcome[field]
    if saved_key is not None:
        idempotency.put(saved_key, response)

    # Grade a sample of real replies AFTER the customer has their answer. It costs money, so only some.
    if outcome["status"] == "replied" and quality.should_sample(quality.SAMPLE_RATE):
        background.add_task(quality.run_online_judge, outcome["request_id"], observe.redact_for_logs(body.email),
                            outcome["reply"], outcome["evidence"])
    return response


@app.post("/feedback")
def post_feedback(body: FeedbackRequest, caller: str = Depends(require_client)):
    """The customer rates a reply. This is the most honest quality signal there is."""
    observe.append_json_line("feedback.jsonl", {
        "time": observe.now_text(),
        "request_id": body.request_id,
        "rating": body.rating,
        "comment": observe.redact_for_logs(body.comment)[:500],
    })
    observe.log(logging.INFO, "feedback_received", judged_request=body.request_id, rating=body.rating)
    return {"saved": True}


@app.get("/approvals")
def get_approvals(admin: str = Depends(require_admin)):
    """Admin: which refunds are waiting for a person to approve them?"""
    return {"waiting": list(guards.PENDING_APPROVALS)}


@app.post("/approvals/{order_id}/approve")
def approve(order_id: str, body: ApprovalRequest, admin: str = Depends(require_admin)):
    """Admin: approve a waiting refund. This is the only way a refund over the limit can be issued."""
    with desk_lock:
        with observe.trace("approval", order_id=order_id, approver=body.approver):
            message = guards.approve_refund(order_id, approver=body.approver)
    return {"result": message}


class KillSwitchRequest(BaseModel):
    on: bool


@app.post("/kill-switch")
def set_kill_switch(body: KillSwitchRequest, admin: str = Depends(require_admin)):
    """Admin: switch the AI off (every email goes to a person) or back on. Something is wrong? Flip this first."""
    state["kill_switch"] = body.on
    observe.audit("kill_switch_changed", on=body.on, changed_by=admin)
    return {"kill_switch": state["kill_switch"]}


@app.get("/metrics")
def get_metrics(admin: str = Depends(require_admin)):
    """Admin: the numbers from the logs, and any alerts that fired."""
    report = metrics.load_report()
    alerts = []
    if report["requests"] > 0:
        alerts = metrics.check_alerts(report, metrics.DEFAULT_LIMITS)
    return {"report": report, "alerts": alerts}
```

## 3. Use it
First, the **health check**. Hosting platforms call this address every few seconds to see whether the service is alive.

In [4]:
show(client.get("/health"))

HTTP 200
{
  "status": "ok",
  "prompt_version": "2026-11-22",
  "code_version": "09ad2de"
}


Now a customer email. The service prints one JSON log line for every request (the `request_finished` line). Notebook 3 explains those lines.

In [5]:
response = client.post("/emails", json={"email": "Hi, can you tell me where my order A1002 is?", "customer_id": "c-101"}, headers=CLIENT)
show(response)

{"time": "2026-11-22T19:15:29+00:00", "level": "INFO", "event": "request_finished", "request_id": "4db6edfc56fc", "status": "ok", "duration_ms": 4931, "model_calls": 6, "cost_usd": 0.00146}


HTTP 200
{
  "request_id": "4db6edfc56fc",
  "status": "replied",
  "reply": "Your order for running shoes is currently in transit and is expected to be delivered on November 26, 2026. If you have any other questions, feel free to ask!",
  "route": "orders",
  "note": "",
  "refunds_issued": [],
  "sent_for_approval": [],
  "cost_usd": 0.00146,
  "latency_ms": 4931
}


Look at the answer. Besides the reply, it includes a **`request_id`** (to find this request in the logs later), the **cost in dollars** and the **latency**. A customer-facing response does not usually include those, but they are very useful to the application that calls us.

A refund below the limit is issued automatically. A big one waits for a human:

In [6]:
show(client.post("/emails", json={"email": "The left earbud on my headphones (order A1001) stopped working. I would like my money back."}, headers=CLIENT))

{"time": "2026-11-22T19:15:34+00:00", "level": "INFO", "event": "audit:refund_issued", "request_id": "e4167ae1dbde", "order_id": "A1001", "amount": 79.0, "approved_by": "agent (under the automatic limit)"}


{"time": "2026-11-22T19:15:35+00:00", "level": "INFO", "event": "request_finished", "request_id": "e4167ae1dbde", "status": "ok", "duration_ms": 5892, "model_calls": 8, "cost_usd": 0.00169}


HTTP 200
{
  "request_id": "e4167ae1dbde",
  "status": "replied",
  "reply": "Your refund of $79.00 has been successfully issued for your order A1001.",
  "route": "refund",
  "note": "",
  "refunds_issued": [
    "A1001"
  ],
  "sent_for_approval": [],
  "cost_usd": 0.00169,
  "latency_ms": 5892
}


In [7]:
show(client.post("/emails", json={"email": "Please refund my 4K television, order A1004. I changed my mind."}, headers=CLIENT))

{"time": "2026-11-22T19:15:40+00:00", "level": "INFO", "event": "audit:refund_held_for_approval", "request_id": "95a052eb3dd0", "order_id": "A1004", "amount": 250.0}


{"time": "2026-11-22T19:15:41+00:00", "level": "INFO", "event": "request_finished", "request_id": "95a052eb3dd0", "status": "ok", "duration_ms": 5813, "model_calls": 8, "cost_usd": 0.001782}


{"time": "2026-11-22T19:15:42+00:00", "level": "INFO", "event": "reply_graded", "request_id": "69040351fdde", "judged_request": "95a052eb3dd0", "score": 5}


{"time": "2026-11-22T19:15:42+00:00", "level": "INFO", "event": "request_finished", "request_id": "69040351fdde", "status": "ok", "duration_ms": 837, "model_calls": 1, "cost_usd": 0.001533}


HTTP 200
{
  "request_id": "95a052eb3dd0",
  "status": "replied",
  "reply": "Your refund request for the 4K television has been sent for approval. Thank you for your patience!",
  "route": "refund",
  "note": "",
  "refunds_issued": [],
  "sent_for_approval": [
    "A1004"
  ],
  "cost_usd": 0.001782,
  "latency_ms": 5813
}


Look at `sent_for_approval`. The $250 refund was **not** issued. A person with an admin key looks at the queue and approves it. Customers' keys cannot do this.

In [8]:
show(client.get("/approvals", headers=ADMIN))

HTTP 200
{
  "waiting": [
    "A1004"
  ]
}


In [9]:
show(client.post("/approvals/A1004/approve", json={"approver": "Priya (support lead)"}, headers=ADMIN))

{"time": "2026-11-22T19:15:42+00:00", "level": "INFO", "event": "audit:refund_approved", "request_id": "45ab0edc7a94", "order_id": "A1004", "approver": "Priya (support lead)"}


{"time": "2026-11-22T19:15:42+00:00", "level": "INFO", "event": "audit:refund_issued", "request_id": "45ab0edc7a94", "order_id": "A1004", "amount": 250.0, "approved_by": "Priya (support lead)"}


{"time": "2026-11-22T19:15:42+00:00", "level": "INFO", "event": "request_finished", "request_id": "45ab0edc7a94", "status": "ok", "duration_ms": 6, "model_calls": 0, "cost_usd": 0.0}


HTTP 200
{
  "result": "Refund of $250.00 issued for order A1004."
}


The approval is recorded **with the approver's name** in an audit trail (notebook 3).

## 4. What FastAPI gives you for free
- **Validation.** Try an empty email. It never reaches the desk.

In [10]:
show(client.post("/emails", json={"email": ""}, headers=CLIENT))

HTTP 422
{
  "detail": [
    {
      "type": "string_too_short",
      "loc": [
        "body",
        "email"
      ],
      "msg": "String should have at least 1 character",
      "input": "",
      "ctx": {
        "min_length": 1
      }
    }
  ]
}


In [11]:
# No key at all
show(client.post("/emails", json={"email": "Hello"}))

HTTP 401
{
  "detail": "Missing or invalid API key."
}


- **A documentation page.** When you run the service for real (`python desk_service.py`), open `http://127.0.0.1:8000/docs`. FastAPI builds an interactive page from your code, where you can try every endpoint in the browser.
- **Speed and concurrency**, when you need them.

## 5. One honest limitation
The desk keeps its state (refunds, the evidence log, the approval queue) in Python lists that all requests share. If two emails were handled at the same moment, they could mix up each other's evidence. That is why `desk_service.py` has a `desk_lock`: **one email at a time**.

That is fine for learning and for a small shop. A real system would keep this state in a **database**, so that many copies of the service could run side by side. Notebook 6 comes back to this.

Next: **what happens when things go wrong?**